# Mutual Fund Overview & Insights
**Name:** Stuti Goyal 

## Stage 0: Project Goal & Problem Statement

**1. Goal.** The goal is to analyse Indian mutual fund schemes and shortlist the 30 with the best balance of return, risk and cost, then present the findings in an interactive dashboard. Everyday investors benefit because it turns a long, confusing list of funds into a short, evidence-based list.

**2. Why balance return and risk.** Ranking by return alone rewards funds that took big risks and just got lucky. For example, a small-cap fund with a very high 3-year return can fall sharply in a bad year, and its expense ratio eats into what the investor actually keeps. A fund with a slightly lower return but low risk and low cost can be the better choice.

**3. Objectives.**
1. Clean the raw data (missing values, wrong formats, bad records).
2. Explore distributions and outliers to understand a typical fund.
3. Normalise the key metrics to a common 0–1 scale.
4. Build a scoring formula and rank all funds.
5. Export the Top 30 to Excel.
6. Build a Power BI dashboard and write the insights.

**4. Role of each tool.** Python (pandas, scikit-learn) cleans the data and calculates the scores. Excel holds the final Top 30 in a format that is easy to share. Power BI turns the results into interactive visuals for a non-technical audience. Each tool does what it is best at.

## Stage 1: Data Cleaning (Python)
### 1.1 Import the dataset
I load the CSV with pandas and check its size, column names and data types.

In [2]:
import pandas as pd
import numpy as np
pd.set_option("display.max_columns", None)

DATA_PATH = "comprehensive_mutual_funds_data.csv"
df = pd.read_csv(DATA_PATH)
print("Rows, columns:", df.shape)

Rows, columns: (814, 20)


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 814 entries, 0 to 813
Data columns (total 20 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   scheme_name    814 non-null    object 
 1   min_sip        814 non-null    int64  
 2   min_lumpsum    814 non-null    int64  
 3   expense_ratio  814 non-null    float64
 4   fund_size_cr   814 non-null    float64
 5   fund_age_yr    814 non-null    int64  
 6   fund_manager   814 non-null    object 
 7   sortino        814 non-null    object 
 8   alpha          814 non-null    object 
 9   sd             814 non-null    object 
 10  beta           814 non-null    object 
 11  sharpe         814 non-null    object 
 12  risk_level     814 non-null    int64  
 13  amc_name       814 non-null    object 
 14  rating         814 non-null    int64  
 15  category       814 non-null    object 
 16  sub_category   814 non-null    object 
 17  returns_1yr    814 non-null    float64
 18  returns_3y

In [4]:
df.head()

,scheme_name,min_sip,min_lumpsum,expense_ratio,fund_size_cr,fund_age_yr,fund_manager,sortino,alpha,sd,beta,sharpe,risk_level,amc_name,rating,category,sub_category,returns_1yr,returns_3yr,returns_5yr
0,Aditya Birla SL Active Debt Multi-Mgr FoF-Dir ...,100,100,0.27,10.0,10,Kaustubh Gupta,0.32,2.24,9.39,0.01,0.24,3,Aditya Birla Sun Life Mutual Fund,3,Other,FoFs Domestic,4.0,6.5,6.9
1,Aditya Birla SL Arbitrage Fund,1000,1000,0.36,4288.0,10,Lovelish Solanki,1.33,1.53,0.72,0.56,1.1,1,Aditya Birla Sun Life Mutual Fund,3,Hybrid,Arbitrage Mutual Funds,5.6,4.8,5.5
2,Aditya Birla SL Asset Allocator FoF-Dir Growth,1000,1000,0.53,157.0,10,Vinod Narayan Bhat,3.44,2.67,10.58,0.67,1.42,5,Aditya Birla Sun Life Mutual Fund,3,Other,FoFs Domestic,2.0,18.9,9.7
3,Aditya Birla SL Bal Bhavishya Yojna – Dir Growth,500,1000,0.76,637.0,4,Atul Penkar,2.18,-6.37,14.99,0.85,0.9,6,Aditya Birla Sun Life Mutual Fund,2,Solution Oriented,Childrens Funds,-0.7,17.1,NaN
4,Aditya Birla SL Balanced Advantage Fund,100,100,0.61,6386.0,10,Mohit Sharma,3.69,1.99,10.38,0.68,1.39,6,Aditya Birla Sun Life Mutual Fund,4,Hybrid,Dynamic Asset Allocation or Balanced Advantage,4.5,18.6,9.7


The dataset has **814 rows and 20 columns**. Key columns:
- `scheme_name`, `amc_name` (fund house), `fund_manager`
- `category` (Equity / Debt / Hybrid / Other / Solution Oriented) and `sub_category`
- `returns_1yr`, `returns_3yr`, `returns_5yr` (% returns)
- `expense_ratio` (% yearly cost), `fund_size_cr` (AUM in ₹ crore), `fund_age_yr`
- `risk_level` (1 = lowest, 6 = highest), `rating` (0 = unrated, 5 = best)
- `min_sip`, `min_lumpsum` (minimum investment amounts)
- `sd`, `beta`, `alpha`, `sharpe`, `sortino` (statistical risk measures)

### 1.2 Remove unnecessary columns

In [5]:
risk_cols = ["sortino", "alpha", "beta", "sharpe"]

# How many "-" placeholders in each advanced ratio?
print((df[risk_cols] == "-").sum())

# Are the beta values sensible?
beta_num = pd.to_numeric(df["beta"], errors="coerce")
print(beta_num.describe())

# Funds with extreme beta (more than 5 away from zero)
df.loc[beta_num.abs() > 5, ["scheme_name", "category", "beta", "sd", "returns_1yr"]]

sortino    23
alpha      42
beta       42
sharpe     23
dtype: int64
count    772.000000
mean       0.822241
std        1.309871
min      -31.860000
25%        0.700000
50%        0.860000
75%        0.970000
max        5.400000
Name: beta, dtype: float64


,scheme_name,category,beta,sd,returns_1yr
85,Bandhan FTP – Sr.179,Debt,5.4,3.98,4.4
89,Bank of India Credit Risk Fund,Debt,-31.86,77.72,130.8
324,ICICI Pru Long Term Equity Fund,Equity,5.04,4.06,-0.2
632,SBI FMP – Series 1 (3668 Days) – Direct Growth,Debt,5.09,3.62,4.8


I drop `sortino`, `alpha`, `beta` and `sharpe`:
- They are advanced ratios that my scoring formula (3-year return, expense ratio, fund age, 1-year return) and the dashboard do not use.
- The cell above shows each has 23–42 "-" placeholders (sortino 23, alpha 42, beta 42, sharpe 23).
- `beta` also has extreme values, from −31.86 to 5.4, while most funds are between 0.7 and 1.0. The −31.86 comes from a credit risk fund that also shows a 130.8% 1-year return, which looks like a data error.

I keep `sd` as my volatility measure. `returns_5yr` is also not needed (167 missing, about 20%), but I keep it until Step 1.4 because I use it to detect bad rows.

In [6]:
df = df.drop(columns=["sortino", "alpha", "beta", "sharpe"])
print("Columns left:", df.shape[1])

Columns left: 16


### 1.3 Identify missing values
`isna()` only finds blank cells, but this dataset also uses a "-" text placeholder. I count both.

In [7]:
missing = pd.DataFrame({
    "blank (NaN)": df.isna().sum(),
    "dash '-'": (df == "-").sum(),
})
missing["total missing"] = missing.sum(axis=1)
missing["% missing"] = (missing["total missing"] / len(df) * 100).round(1)
missing[missing["total missing"] > 0]

,blank (NaN),dash '-',total missing,% missing
sd,0,24,24,2.9
returns_3yr,21,0,21,2.6
returns_5yr,167,0,167,20.5


Missing data: `returns_5yr` (167 blank, 20.5%), `returns_3yr` (21 blank, 2.6%) and `sd` (24 "-" values, 2.9%). These are mostly new funds with too little history. **Strategy:** I do not fill them with an average, because that would invent performance numbers. I keep these funds for the dashboard and exclude them only from ranking, where a 3-year return is required.

### 1.4 Standardise numeric formats
Issues found: (1) `sd` is stored as text because of "-"; (2) stray spaces in text columns; (3) zeros that really mean "not available" (SIP = 0 means no SIP offered; expense ratio = 0 for fixed-maturity plans is not realistic); (4) identical return values copied across unrelated funds.

In [8]:
# (1) "-" -> NaN, then convert to number
df["sd"] = pd.to_numeric(df["sd"].replace("-", np.nan), errors="coerce")

# (2) trim spaces in text columns
for c in ["scheme_name", "fund_manager", "amc_name", "category", "sub_category"]:
    df[c] = df[c].str.strip()

# (3) zeros that mean "not available" -> NaN
for c in ["min_sip", "min_lumpsum", "expense_ratio"]:
    print(c, "zeros:", (df[c] == 0).sum())
    df[c] = df[c].replace(0, np.nan)

# (4) same (1yr, 3yr, 5yr) triple shared by 3+ funds across different categories
key = ["returns_1yr", "returns_3yr", "returns_5yr"]
groups = (df.dropna(subset=key).groupby(key)
            .agg(funds=("scheme_name", "size"), categories=("category", "nunique"))
            .reset_index())
suspect = groups[(groups["funds"] >= 3) & (groups["categories"] > 1)]
print(suspect)
bad = df.set_index(key).index.isin(suspect.set_index(key).index)
print(df.loc[bad, ["scheme_name", "category"] + key])

df.loc[bad, ["returns_1yr", "returns_3yr"]] = np.nan
df = df.drop(columns=["returns_5yr"])
print("Funds with invalid returns set to NaN:", bad.sum())

min_sip zeros: 42
min_lumpsum zeros: 2
expense_ratio zeros: 2
     returns_1yr  returns_3yr  returns_5yr  funds  categories
81          -0.7         33.7         11.3      5           2
164          1.7         32.4         13.3      9           4
                                     scheme_name           category  \
225              HDFC Capital Builder Value Fund             Equity   
267                     HSBC Corporate Bond Fund               Debt   
272                      HSBC Equity Hybrid Fund             Hybrid   
275                         HSBC Flexi Debt Fund               Debt   
325  ICICI Pru Long Term Wealth Enhancement Fund             Equity   
340                    ICICI Pru Retirement Fund  Solution Oriented   
355                   IDBI Equity Advantage Fund             Equity   
461                      L&T Focused Equity Fund             Equity   
487        Mahindra Manulife Equity Savings Fund             Hybrid   
516             Motilal Oswal Equity Hybri

Two return combinations (1.7 / 32.4 / 13.3 and −0.7 / 33.7 / 11.3) repeat across **14 unrelated funds**, including overnight and corporate bond funds. A bond fund cannot return 32% a year, so these look like copy-paste errors. I set their returns to NaN so they cannot distort the ranking. 25 scheme names repeat, but their numbers differ (different plans or series), so I keep them.